# Fine-tune a procurement extractor — Colab run (synthetic data)

Trains **Qwen2.5-7B-Instruct** (QLoRA, 4-bit) on 2,000 synthetic quotations
to output strict JSON with anomaly flags, then compares it against the base
model on 100 held-out quotes.

**Setup once:** Runtime → Change runtime type → **T4 GPU**.
**Then:** Run all cells (Ctrl+F9). Cell 3 pulls the synthetic data straight
from the public repo (no upload needed). Expect ~45-70 minutes total.
All data is fictional — nothing real leaves the machine.


In [ ]:
import subprocess, sys, time
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:400])

In [ ]:
# Install the training stack (one-time per runtime, ~2-4 min)
import importlib.util, shutil, subprocess, sys
print("Python:", sys.version.split()[0], "| exe:", sys.executable)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "soup-cli[train]", "accelerate", "bitsandbytes"],
                   capture_output=True, text=True)
print("pip exit:", r.returncode)
if r.returncode != 0:
    print((r.stdout or "")[-600:])
    print((r.stderr or "")[-1500:])
if importlib.util.find_spec("soup_cli") is None:
    print("soup_cli not importable -> retrying with --ignore-requires-python")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "--ignore-requires-python",
                        "soup-cli[train]", "accelerate", "bitsandbytes"],
                       capture_output=True, text=True)
    print("pip exit:", r.returncode)
    if r.returncode != 0:
        print((r.stdout or "")[-600:])
        print((r.stderr or "")[-1500:])
print("soup_cli importable:", importlib.util.find_spec("soup_cli") is not None)
print("soup CLI on PATH:", shutil.which("soup"))


In [ ]:
# Data comes from the public repo (all synthetic). Always pull latest config.
import os, subprocess, sys
REPO = "https://github.com/rubinagentagi-tech/procurement-extractor-finetune.git"
DIR = "procurement-extractor-finetune"
if os.path.isdir(DIR):
    r = subprocess.run(["git", "-C", DIR, "pull", "--ff-only"], capture_output=True, text=True)
else:
    r = subprocess.run(["git", "clone", "--depth", "1", REPO], capture_output=True, text=True)
print((r.stdout or "")[-400:], (r.stderr or "")[-300:])
os.chdir(DIR)
print("Files:", sorted(f for f in os.listdir(".") if not f.startswith(".")))
print("Data rows:", sum(1 for _ in open("data/train.jsonl")))


In [ ]:
import shutil, os
# 1-epoch config for the free tier (2-epoch config lives in soup.yaml for paid GPUs)
shutil.copy("colab-soup.yaml", "soup.yaml")
print("Using config:")
print(open("soup.yaml").read())

In [ ]:
print("Starting training (streaming output below)...\n")
import shutil, subprocess, sys
cmd = (["soup", "train", "-c", "soup.yaml", "--yes"] if shutil.which("soup")
       else [sys.executable, "-m", "soup_cli.cli", "train", "-c", "soup.yaml", "--yes"])
print("cmd:", " ".join(cmd), flush=True)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
tail = []
for line in p.stdout:
    print(line.rstrip(), flush=True)
    tail.append(line)
    if len(tail) > 500:
        tail = tail[-500:]
rc = p.wait()
print("EXIT:", rc)
if rc != 0:
    print("\\n--- last 40 lines of the failed run ---")
    print("".join(tail[-40:]))


In [ ]:
import os
# show the loss curve if logged
logd = "output"
if os.path.isdir(logd):
    for f in sorted(os.listdir(logd))[:20]: print(" ", f)
else:
    print("no output dir")

In [ ]:
print("Evaluating baseline vs fine-tuned (100 quotes)...")
import subprocess, sys
r = subprocess.run([sys.executable, "eval_check.py",
                    "--base", "Qwen/Qwen2.5-7B-Instruct",
                    "--adapter", "./output"],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-2000:])

## Save your proof
The next cell zips the adapter + config + eval log and downloads it to your
computer. Keep it — it is your *"I trained a model"* artifact (adapter
~150 MB, merges back into Qwen2.5-7B-Instruct with `soup merge`).

In [ ]:
import zipfile, os
with zipfile.ZipFile("colab-trial-results.zip", "w") as z:
    for root, _, fs in os.walk("output"):
        for f in fs:
            p = os.path.join(root, f)
            z.write(p, p)
    for f in ("soup.yaml", "colab-soup.yaml"):
        if os.path.exists(f): z.write(f)
from google.colab import files
files.download("colab-trial-results.zip")
print("Downloaded: colab-trial-results.zip (adapter + config + logs)")